# Семинар 1. Корпуса и предобработка текста

**Курс:** «Интеллектуальный анализ текстов: от статистических моделей языка до языковых агентов». Семинар относится к Лекции 1 «Введение в курс. Статистическое моделирование языка».

## Цель

Освоить базовый пайплайн предобработки текста и эмпирически проверить законы Ципфа и Хипса на реальных корпусах.

## Задачи семинара

1. Разобрать этапы предобработки: нормализация, токенизация, лемматизация/стемминг, фильтрация стоп-слов.
2. Сравнить простые токенизаторы (по пробелам, регулярные выражения) с библиотечными (razdel, spaCy, nltk).
3. Построить частотный словарь корпуса и ранговое распределение частот.
4. Проверить закон Ципфа (лог-лог график) и закон Хипса (рост словаря от объёма текста).

## Ход занятия

1. Загрузка корпуса новостей Lenta.ru (через `datasets` с HuggingFace Hub, потоковый режим, маленький сэмпл).
2. Реализация пайплайна предобработки на Python (`re`, простой токенизатор; опционально `razdel`).
3. Подсчёт частот, построение таблицы топ-N слов.
4. Построение графиков Ципфа и Хипса, обсуждение отклонений на хвостах распределения.
5. Сравнение статистик корпуса с идеальным законом Ципфа ($\alpha \approx 1$) как baseline.


## Краткая теория

### 1. Предобработка текста

Типовой пайплайн превращения «сырого» текста в последовательность токенов для статистических моделей:

1. **Нормализация** — приведение текста к единообразному виду: нижний регистр, унификация кавычек/дефисов, иногда снятие ударений.
2. **Токенизация** — разбиение текста на минимальные единицы (токены): слова, знаки пунктуации, числа. Простейший вариант — разбиение по пробелам; более аккуратные варианты используют регулярные выражения или специализированные библиотечные токенизаторы (`razdel` для русского, `nltk`, `spaCy`), которые корректно обрабатывают сокращения, дефисные слова, знаки пунктуации.
3. **Лемматизация/стемминг** — приведение словоформы к начальной форме (лемме) или обрезание до основы (стем). Для русского языка лемматизация особенно важна из-за богатой морфологии (падежи, числа, времена).
4. **Фильтрация стоп-слов** — удаление высокочастотных служебных слов (предлоги, союзы, частицы), которые обычно не несут различительной информации для задач типа классификации или тематического моделирования.

### 2. Частотный словарь и ранговое распределение

Для корпуса токенов строится словарь $V$ с частотами $C(w)$ для каждого слова $w \in V$. Ранжируя слова по невозрастанию частоты, слову присваивается ранг $r \in \{1, 2, \dots, |V|\}$ (ранг 1 — самое частое слово).

### 3. Закон Ципфа (Zipf's law)

**Утверждение.** Частота $f(r)$ слова с рангом $r$ приближённо подчиняется степенному закону:
$$
f(r) \approx \frac{C}{r^{\alpha}}, \qquad \alpha \approx 1.
$$
В логарифмических координатах это линейная зависимость:
$$
\log f(r) = \log C - \alpha \log r,
$$
поэтому показатель $\alpha$ оценивается методом наименьших квадратов (МНК) по линейной регрессии $\log f$ на $\log r$.

**Интуиция.** Небольшое число слов (предлоги, союзы) покрывает бо́льшую часть текста, а «хвост» словаря состоит из огромного числа редких слов — это и объясняет разреженность данных, из-за которой в языковых моделях нужны сглаживания (см. Семинар 2).

### 4. Закон Хипса (Heaps' law)

**Утверждение.** Размер словаря $|V(N)|$ (число уникальных слов) как функция от объёма текста $N$ (в токенах) растёт сублинейно:
$$
|V(N)| \approx K \cdot N^{\beta}, \qquad 0 < \beta < 1,
$$
что также линейно в лог-лог координатах: $\log|V(N)| = \log K + \beta \log N$.

**Интуиция.** С ростом текста встречаются всё новые слова, но темп появления новых слов замедляется — вначале почти каждое слово новое, а затем корпус всё чаще «повторяет» уже встреченные слова.

### 5. Метрика сравнения с идеальным законом

В качестве **baseline** для эмпирического распределения используется идеальный закон Ципфа с $\alpha = 1$ (без подгонки). Качество приближения оценивается через коэффициент детерминации $R^2$ линейной регрессии в лог-лог координатах:
$$
R^2 = 1 - \frac{\sum_i (\log f_i - \widehat{\log f_i})^2}{\sum_i (\log f_i - \overline{\log f})^2}.
$$
Чем ближе $R^2$ к 1, тем точнее эмпирическое распределение описывается степенным законом.


## Подготовка

Загрузим небольшой сэмпл русского новостного корпуса **Lenta.ru** через библиотеку `datasets` (HuggingFace Hub, потоковый режим `streaming=True` — без скачивания всего датасета). Если Hub недоступен (нет сети / ограничение окружения), используется offline-фолбэк: встроенный небольшой корпус русских новостных предложений.

Все операции фиксируют `random.seed`/`np.random.seed` для воспроизводимости.


In [ ]:
import re
import json
import random
import warnings
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

N_DOCS = 400  # небольшой сэмпл -- работает на CPU за секунды/десятки секунд

print("Импорты готовы, SEED =", SEED)


In [ ]:
def load_lenta_sample(n_docs=N_DOCS):
    """Загружает n_docs новостных текстов Lenta.ru через HuggingFace datasets (streaming).
    При любой ошибке (нет сети, датасет недоступен) откатывается на offline-корпус.
    """
    try:
        from datasets import load_dataset
        ds = load_dataset("zloelias/lenta-ru", split="train", streaming=True)
        texts = []
        for i, row in enumerate(ds):
            if i >= n_docs:
                break
            t = (row.get("text") or "").strip()
            if t:
                texts.append(t)
        if len(texts) >= 50:
            return texts, "lenta_ru (HuggingFace, streaming)"
    except Exception as e:
        print("Не удалось загрузить Lenta.ru с Hub:", repr(e))

    # ---- offline fallback: небольшой встроенный корпус русских новостных предложений ----
    fallback_sentences = [
        "Правительство объявило о новых мерах поддержки малого бизнеса в этом году.",
        "Центральный банк снизил ключевую ставку на фоне замедления инфляции.",
        "Учёные представили новое исследование о влиянии климата на урожайность.",
        "Компания сообщила о рекордной прибыли по итогам последнего квартала.",
        "Власти города утвердили план реконструкции центральных улиц.",
        "Спортивная сборная одержала победу в решающем матче турнира.",
        "Министерство науки анонсировало запуск новой образовательной программы.",
        "Аналитики прогнозируют рост цен на энергоносители в следующем месяце.",
        "Разработчики выпустили обновление популярного приложения для смартфонов.",
        "Депутаты приняли закон о регулировании цифровых финансовых активов.",
        "Врачи предупреждают о росте заболеваемости в осенний период.",
        "Инвесторы проявляют интерес к акциям технологических компаний.",
        "Городская администрация запустила программу благоустройства парков.",
        "Эксперты обсудили перспективы развития искусственного интеллекта.",
        "Автопроизводитель представил новую модель электромобиля на выставке.",
        "Пограничная служба сообщила об усилении контроля на границе.",
        "Судебное разбирательство по делу о мошенничестве продолжается второй месяц.",
        "Фермеры собрали рекордный урожай зерновых в этом сезоне.",
        "Банк запустил новый сервис онлайн-кредитования для предпринимателей.",
        "Журналисты выяснили подробности сделки между крупными корпорациями.",
    ]
    random.shuffle(fallback_sentences)
    # умножаем и слегка перемешиваем, чтобы получить корпус разумного объёма для демонстрации
    texts = []
    for i in range(n_docs):
        texts.append(fallback_sentences[i % len(fallback_sentences)])
    print("ВНИМАНИЕ: используется offline-fallback. Корпус вырожден (20 предложений,")
    print("повторённых по кругу), поэтому количественные проверки законов Ципфа/Хипса")
    print("и сравнительные выводы НЕДЕЙСТВИТЕЛЬНЫ. Для содержательных результатов")
    print("используйте полный корпус Lenta.ru (нужен доступ к HuggingFace Hub).")
    return texts, "offline fallback corpus (встроенные новостные предложения)"


raw_texts, corpus_source = load_lenta_sample(N_DOCS)
print(f"Источник корпуса: {corpus_source}")
print(f"Загружено документов: {len(raw_texts)}")
print("Пример документа:", raw_texts[0][:200])


## Демо-задача 1. Сравнение токенизаторов

**Постановка.** Сравним три подхода к токенизации на предложениях с сокращениями и пунктуацией:
1. **Baseline** — наивное разбиение по пробелам (`str.split()`).
2. **Регулярное выражение** — токенизация по regex, отделяющая пунктуацию от слов.
3. **Библиотечный токенизатор** — `razdel` (если доступен), иначе собственная regex-реализация с более полным покрытием случаев (дефисные слова, многоточия).

**Метрика.** Число токенов на предложение и качественное сравнение разбиения (совпадение/несовпадение границ токенов).


In [ ]:
SAMPLE_SENTENCES = [
    "Санкт-Петербург — второй по величине город России.",
    "Он сказал: «Приду в 9:30, т.е. вовремя!»...",
    "ООО «Ромашка» увеличило прибыль на 12,5% за 2023 г.",
    "Не думаю, что это по-настоящему важно — впрочем, кто знает?",
    "МГУ им. М.В. Ломоносова объявил набор на 2024/2025 учебный год.",
]

def tokenize_whitespace(text):
    """Baseline: наивное разбиение по пробелам."""
    return text.split()

def tokenize_regex(text):
    """Регулярное выражение: слова (включая дефисные) отдельно от пунктуации."""
    return re.findall(r"[\w\-]+|[^\w\s]", text, flags=re.UNICODE)

def tokenize_library(text):
    """Библиотечный токенизатор: razdel, если установлен, иначе regex-fallback
    с более аккуратной обработкой сокращений и многоточий."""
    try:
        from razdel import tokenize as razdel_tokenize
        return [t.text for t in razdel_tokenize(text)]
    except ImportError:
        # fallback: чуть более аккуратный regex (не разбивает "т.е.", числа с запятой)
        pattern = r"\d+[.,]\d+|[а-яА-ЯёЁa-zA-Z]+\.(?:[а-яА-ЯёЁa-zA-Z]+\.)+|[\w\-]+|[^\w\s]"
        return re.findall(pattern, text, flags=re.UNICODE)


rows = []
for s in SAMPLE_SENTENCES:
    t_ws = tokenize_whitespace(s)
    t_re = tokenize_regex(s)
    t_lib = tokenize_library(s)
    rows.append((len(t_ws), len(t_re), len(t_lib)))

print(f"{'Предложение':<55}{'пробел':>8}{'regex':>8}{'lib':>8}")
for s, (n_ws, n_re, n_lib) in zip(SAMPLE_SENTENCES, rows):
    short = s if len(s) <= 52 else s[:49] + "..."
    print(f"{short:<55}{n_ws:>8}{n_re:>8}{n_lib:>8}")

print("\nПример разбиения регулярным выражением:")
print(tokenize_regex(SAMPLE_SENTENCES[1]))
print("\nПример разбиения библиотечным токенизатором:")
print(tokenize_library(SAMPLE_SENTENCES[1]))


**Сравнение с baseline.**

| Токенизатор | Среднее число токенов/предложение | Особенности |
|---|---|---|
| Пробельный (baseline) | наименьшее (сравните с выводом ячейки выше) | не отделяет знаки пунктуации: «9:30,» — один токен, поэтому токенов меньше, но их границы не смысловые |
| Regex | наибольшее | отделяет пунктуацию, но дробит сокращения («т.е.» → «т», «.», «е», «.») — искусственно завышает число токенов |
| Библиотечный (razdel/regex-fallback) | промежуточное (меньше, чем у regex) | корректно обрабатывает сокращения и составные конструкции |

**Вывод.** Наивная пробельная токенизация даёт наименьшее число токенов из-за «слипания» пунктуации со словами (границы токенов при этом не смысловые), а простой regex, наоборот, даёт наибольшее число токенов, искусственно раздробляя сокращения. Библиотечные токенизаторы (или их regex-аппроксимации) занимают промежуточное положение и дают наиболее корректное для дальнейшего статистического анализа разбиение. Точные средние значения сравните с выводом ячейки выше.


## Демо-задача 2. Построение частотного словаря корпуса

**Постановка.** Применим пайплайн предобработки (нормализация → токенизация → фильтрация) ко всему загруженному корпусу и построим частотный словарь.

**Baseline.** В качестве baseline предобработки используем минимальную нормализацию: только нижний регистр без токенизации по словам-границам (пробельное разбиение). Основная модель — regex-токенизация с очисткой пунктуации.

**Метрика.** Покрытие корпуса топ-N словами: доля токенов корпуса, приходящаяся на N самых частых слов.


In [ ]:
TOKEN_RE = re.compile(r"[а-яёa-z]+", re.UNICODE)

def preprocess_baseline(texts):
    """Baseline: нижний регистр + пробельное разбиение (без чистки пунктуации)."""
    tokens = []
    for t in texts:
        tokens.extend(t.lower().split())
    return tokens

def preprocess_main(texts):
    """Основной пайплайн: нижний регистр -> regex-токенизация (только буквы) ->
    фильтрация токенов короче 2 символов (шум/опечатки/одиночные буквы)."""
    tokens = []
    for t in texts:
        t_low = t.lower()
        toks = TOKEN_RE.findall(t_low)
        tokens.extend([tok for tok in toks if len(tok) >= 2])
    return tokens


tokens_baseline = preprocess_baseline(raw_texts)
tokens_main = preprocess_main(raw_texts)

freq_baseline = Counter(tokens_baseline)
freq_main = Counter(tokens_main)

print(f"Baseline: {len(tokens_baseline)} токенов, {len(freq_baseline)} уникальных")
print(f"Основной пайплайн: {len(tokens_main)} токенов, {len(freq_main)} уникальных")

print("\nТоп-15 слов (основной пайплайн):")
for w, c in freq_main.most_common(15):
    print(f"  {w:<20}{c}")


In [ ]:
def top_n_coverage(freq_counter, n):
    total = sum(freq_counter.values())
    top_n_sum = sum(c for _, c in freq_counter.most_common(n))
    return top_n_sum / total if total else 0.0

coverage_table = []
for n in [10, 50, 100, 500]:
    cov_base = top_n_coverage(freq_baseline, n)
    cov_main = top_n_coverage(freq_main, n)
    coverage_table.append((n, cov_base, cov_main))

print(f"{'top-N':>8}{'baseline (пробел)':>22}{'основной (regex+фильтр)':>28}")
for n, cb, cm in coverage_table:
    print(f"{n:>8}{cb:>21.1%}{cm:>28.1%}")


**Вывод.** Небольшое число самых частых слов (предлоги, союзы, частицы) покрывает значительную долю корпуса уже при $N=100$–500, что является прямым следствием закона Ципфа (проверяется в следующей задаче). Основной пайплайн с чисткой пунктуации даёт более осмысленный словарь: baseline «загрязняет» топ случайными слипаниями слов со знаками пунктуации.


## Демо-задача 3. Проверка закона Ципфа

**Постановка.** Построим ранговое распределение частот слов основного пайплайна, оценим показатель $\alpha$ методом наименьших квадратов на лог-лог графике и сравним с **baseline** — идеальным законом Ципфа $\alpha = 1$.

**Метрика.** Коэффициент детерминации $R^2$ линейной регрессии $\log f(r)$ на $\log r$ (см. формулу в теории) — отдельно для оценённого $\hat\alpha$ и для фиксированного baseline $\alpha=1$.

**Замечание о диапазоне рангов.** МНК-оценка $\hat\alpha$ чувствительна к выбору диапазона рангов: «голова» распределения (несколько самых частых слов) и «хвост» (слова с частотой 1–2, где лог-лог график ступенчатый) систематически отклоняются от степенного закона. На практике $\alpha$ часто оценивают по среднему диапазону рангов (например, отбросив первые ~10 рангов и хвост с частотой $< 3$); ниже для простоты используется весь диапазон — сравните, как меняется $\hat\alpha$ при усечении (см. также задачу 1).


In [ ]:
ranked = freq_main.most_common()
ranks = np.arange(1, len(ranked) + 1)
freqs = np.array([c for _, c in ranked], dtype=float)

log_r = np.log(ranks)
log_f = np.log(freqs)

# МНК: log_f = log_C - alpha * log_r  =>  линейная регрессия (slope, intercept)
# NB: для простоты регрессия строится по ВСЕМ рангам; более аккуратная оценка
# исключает голову (первые ~10 рангов) и хвост (частоты < 3), где закон нарушается
slope, intercept = np.polyfit(log_r, log_f, 1)
alpha_hat = -slope
C_hat = np.exp(intercept)

def r_squared(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot if ss_tot > 0 else 0.0

# Модель с оценённым alpha_hat
log_f_pred_fitted = intercept + slope * log_r
r2_fitted = r_squared(log_f, log_f_pred_fitted)

# Baseline: идеальный закон Ципфа с alpha=1, C выбрана так, чтобы совпасть в r=1
C_baseline = freqs[0]
log_f_pred_baseline = np.log(C_baseline) - 1.0 * log_r
r2_baseline = r_squared(log_f, log_f_pred_baseline)

print(f"Оценённый показатель Ципфа: alpha_hat = {alpha_hat:.3f}")
print(f"R^2 (модель с alpha_hat)   = {r2_fitted:.4f}")
print(f"R^2 (baseline, alpha=1)    = {r2_baseline:.4f}")


In [ ]:
plt.figure(figsize=(6, 4.5))
plt.loglog(ranks, freqs, ".", markersize=3, alpha=0.5, label="эмпирические частоты")
plt.loglog(ranks, np.exp(log_f_pred_fitted), "-", color="tab:red",
           label=f"МНК-приближение, alpha={alpha_hat:.2f}")
plt.loglog(ranks, np.exp(log_f_pred_baseline), "--", color="tab:green",
           label="идеальный закон Ципфа, alpha=1")
plt.xlabel("ранг r (лог. шкала)")
plt.ylabel("частота f(r) (лог. шкала)")
plt.title("Закон Ципфа: частота слова от ранга")
plt.legend()
plt.tight_layout()
plt.savefig("zipf_law.png", dpi=100)
plt.show()
print("График сохранён в zipf_law.png")


**Таблица сравнения с baseline.**

| Модель | $\alpha$ | $R^2$ | Комментарий |
|---|---|---|---|
| Baseline (идеальный закон Ципфа) | 1.00 (фиксирован) | см. вывод кода выше | не учитывает специфику корпуса |
| МНК-оценка (основная модель) | $\hat\alpha$ (см. вывод кода выше) | см. вывод кода выше | подгоняется под данные |

**Вывод.** Эмпирическое ранговое распределение частот слов корпуса приближённо линейно в лог-лог координатах, что подтверждает закон Ципфа; отклонения на «хвосте» (крайне редкие слова, ранги ближе к $|V|$) — типичное явление, связанное с конечностью корпуса и шумом единичных наблюдений.


## Демо-задача 4. Проверка закона Хипса

**Постановка.** Построим кривую роста словаря $|V(N)|$ на растущих префиксах корпуса (по числу обработанных токенов $N$) и оценим показатель $\beta$ методом наименьших квадратов в лог-лог координатах.

**Baseline.** В качестве baseline сравнения используем линейный рост словаря ($\beta_{\text{baseline}}=1$, т.е. предположение «каждое новое слово — уникально»), который является верхней границей и заведомо не подтвердится на реальном тексте.

**Метрика.** Оценённый показатель $\hat\beta$ и $R^2$ приближения в лог-лог координатах.


In [ ]:
# Строим кривую Хипса на растущих префиксах токенизированного корпуса
step = max(1, len(tokens_main) // 200)  # ~200 точек для быстрого построения
prefix_lengths = list(range(step, len(tokens_main) + 1, step))

vocab_sizes = []
seen = set()
idx = 0
for target_n in prefix_lengths:
    while idx < target_n:
        seen.add(tokens_main[idx])
        idx += 1
    vocab_sizes.append(len(seen))

N_arr = np.array(prefix_lengths, dtype=float)
V_arr = np.array(vocab_sizes, dtype=float)

log_N = np.log(N_arr)
log_V = np.log(V_arr)

slope_h, intercept_h = np.polyfit(log_N, log_V, 1)
beta_hat = slope_h
K_hat = np.exp(intercept_h)

log_V_pred = intercept_h + slope_h * log_N
r2_heaps = r_squared(log_V, log_V_pred)

# Baseline: линейный рост словаря (beta=1), K подобрана по первой точке
K_baseline = V_arr[0] / N_arr[0]
log_V_pred_baseline = np.log(K_baseline) + 1.0 * log_N
r2_heaps_baseline = r_squared(log_V, log_V_pred_baseline)

print(f"Оценённый показатель Хипса: beta_hat = {beta_hat:.3f}")
print(f"R^2 (модель с beta_hat)     = {r2_heaps:.4f}")
print(f"R^2 (baseline, beta=1)      = {r2_heaps_baseline:.4f}")


In [ ]:
plt.figure(figsize=(6, 4.5))
plt.plot(N_arr, V_arr, ".", markersize=3, alpha=0.6, label="эмпирический рост словаря")
plt.plot(N_arr, np.exp(log_V_pred), "-", color="tab:red",
         label=f"МНК-приближение, beta={beta_hat:.2f}")
plt.xlabel("объём корпуса N (токенов)")
plt.ylabel("размер словаря |V(N)|")
plt.title("Закон Хипса: рост словаря от объёма корпуса")
plt.legend()
plt.tight_layout()
plt.savefig("heaps_law.png", dpi=100)
plt.show()
print("График сохранён в heaps_law.png")


**Таблица сравнения с baseline.**

| Модель | $\beta$ | $R^2$ | Комментарий |
|---|---|---|---|
| Baseline (линейный рост, $\beta=1$) | 1.00 (фиксирован) | см. вывод кода выше | верхняя граница, нереалистична |
| МНК-оценка (основная модель) | $\hat\beta$ (см. вывод кода выше) | см. вывод кода выше | сублинейный рост, типичное значение $0.4$–$0.7$ |

**Вывод.** Рост словаря замедляется с увеличением объёма текста ($\hat\beta < 1$): в начале корпуса почти каждое слово новое, а затем всё чаще встречаются уже виденные слова. Показатель $\beta$ и показатель Ципфа $\alpha$ теоретически связаны (для идеализированного ципфовского корпуса выполняется приближение $\beta \approx 1/\alpha$; вывод этой связи выходит за рамки семинара) — оба явления — следствие одной и той же неоднородности частот слов естественного языка.


## Задачи для самостоятельного решения

Ниже — 5 задач из списка задач семинара в силлабусе (адаптированные под данный корпус и переменные, уже созданные выше). Для каждой задачи: посчитайте метрику, сравните с указанным baseline, сделайте вывод (1–2 предложения) в отдельной markdown-ячейке после кода.


### Задача 1. Оценка показателя Ципфа методом наименьших квадратов

По корпусу Lenta.ru (переменная `tokens_main`) постройте лог-лог график частота–ранг для **другого** варианта предобработки (например, без фильтрации коротких токенов, или с фильтрацией стоп-слов) и оцените $\hat\alpha$ методом наименьших квадратов. Сравните с `alpha_hat`, полученным в демо-задаче 3 (**baseline** для сравнения — уже посчитанный $\hat\alpha$ из демо-задачи).

**Ожидаемый диапазон результата:** $\hat\alpha \in [0.7,\ 1.3]$; отличие от baseline демо-задачи обычно не превышает $\pm0.2$.


In [ ]:
# TODO: реализуйте альтернативный пайплайн предобработки (например, с фильтрацией стоп-слов)
# TODO: постройте частотный словарь alt_freq = Counter(...)
# TODO: постройте лог-лог график ранг/частота, оцените alpha методом МНК (np.polyfit)
# TODO: сравните с alpha_hat (baseline из демо-задачи 3), выведите разницу
# TODO: сделайте вывод в markdown-ячейке ниже

alt_freq = None  # замените на Counter(...)
alpha_hat_alt = None  # замените на оценку МНК


**Вывод:** _(заполните после выполнения кода выше)_

### Задача 2. Сравнение токенизаторов на предложениях с сокращениями

Возьмите 5 собственных предложений с сокращениями и знаками пунктуации (можно расширить `SAMPLE_SENTENCES` из демо-задачи 1) и сравните три токенизатора: пробельный (**baseline**), regex, библиотечный (`tokenize_library`). Зафиксируйте различия в числе токенов на каждое предложение в виде таблицы.

**Ожидаемый диапазон результата:** библиотечный токенизатор в среднем даёт на 10–30% меньше токенов, чем regex, из-за корректной обработки сокращений; baseline (пробельный) обычно даёт наименьшее число токенов за счёт «слипания» пунктуации со словами, но не по смысловым границам.


In [ ]:
# TODO: придумайте 5 новых предложений с сокращениями/пунктуацией
MY_SENTENCES = [
    # "...",
]

# TODO: примените tokenize_whitespace (baseline), tokenize_regex, tokenize_library к каждому предложению
# TODO: соберите таблицу (список строк) с числом токенов для каждого метода
# TODO: сравните средние числа токенов между методами
# TODO: сделайте вывод в markdown-ячейке ниже


**Вывод:** _(заполните после выполнения кода выше)_

### Задача 3. Кривая Хипса и оценка показателя $\beta$

Постройте кривую Хипса (размер словаря от объёма корпуса в токенах) на растущих префиксах текста для baseline-предобработки (`tokens_baseline`, пробельное разбиение без чистки) и оцените показатель $\beta$. Сравните с $\hat\beta$, полученным в демо-задаче 4 для основного пайплайна (это и есть **baseline** для сравнения).

**Ожидаемый диапазон результата:** $\hat\beta_{\text{baseline\_preprocess}} \in [0.3,\ 0.8]$; из-за «загрязнённого» словаря (пунктуация слипается со словами) размер словаря на пробельном разбиении обычно растёт быстрее (выше $\beta$), чем на очищенном корпусе.


In [ ]:
# TODO: постройте prefix_lengths и vocab_sizes для tokens_baseline (аналогично демо-задаче 4)
# TODO: оцените beta методом МНК в лог-лог координатах (np.polyfit)
# TODO: сравните с beta_hat из демо-задачи 4 (основной пайплайн) -- это baseline для сравнения
# TODO: постройте график (опционально) и сделайте вывод в markdown-ячейке ниже


**Вывод:** _(заполните после выполнения кода выше)_

### Задача 4. Доля словаря, покрывающая 90% токенов корпуса

Оцените долю уникальных слов словаря `freq_main`, которая покрывает 90% всех токенов корпуса (проверка гипотезы «длинного хвоста»). Сравните с **baseline**-предположением равномерного распределения частот (если бы все слова были равновероятны, для покрытия 90% токенов потребовалось бы 90% словаря).

**Ожидаемый диапазон результата:** реальная доля словаря, покрывающая 90% токенов, обычно составляет всего 20–50% от размера словаря — существенно меньше, чем 90% при равномерном распределении (baseline).


In [ ]:
# TODO: используя freq_main.most_common(), посчитайте кумулятивную сумму частот
# TODO: найдите минимальное число слов N90, для которого кумулятивная сумма >= 0.9 * общее число токенов
# TODO: посчитайте долю N90 / |V| (размер словаря)
# TODO: сравните с baseline = 0.9 (гипотетическая доля при равномерном распределении)
# TODO: сделайте вывод в markdown-ячейке ниже


**Вывод:** _(заполните после выполнения кода выше)_

### Задача 5. Влияние нормализации на оценённый показатель Ципфа

Сравните показатель $\hat\alpha$, оценённый на корпусе **без** приведения к нижнему регистру, с $\hat\alpha$ основной модели (нижний регистр применён, демо-задача 3 — **baseline** для сравнения). Обсудите, почему регистр влияет (или не влияет) на оценку.

**Ожидаемый диапазон результата:** отличие показателей $\hat\alpha$ обычно небольшое (в пределах 0.05–0.15), но без приведения к нижнему регистру словарь искусственно увеличивается (слова с заглавной и строчной буквы считаются разными токенами), что чуть смещает распределение частот и показатель $\alpha$.


In [ ]:
# TODO: постройте пайплайн предобработки БЕЗ .lower() (сохраняя регистр)
# TODO: постройте частотный словарь, оцените alpha методом МНК
# TODO: сравните полученный alpha с alpha_hat (baseline, из демо-задачи 3, с нижним регистром)
# TODO: сделайте вывод в markdown-ячейке ниже


**Вывод:** _(заполните после выполнения кода выше)_

## Чек-лист оценивания

При проверке решений семинара учитывайте следующие критерии:

- [ ] **Прозрачная подготовка данных**: этапы предобработки явно разделены (нормализация → токенизация → фильтрация); все сравниваемые варианты предобработки применяются к одному и тому же сэмплу; объём сэмпла указан и обоснован. (Деления на train/test на этом семинаре нет — модель не обучается.)
- [ ] **Наличие baseline**: для каждой задачи есть явное сравнение с базовым решением (наивная токенизация, идеальный закон Ципфа $\alpha=1$, идеальный линейный рост словаря $\beta=1$ или равномерное распределение частот).
- [ ] **Выбор метрики**: используется адекватная метрика (число токенов, покрытие топ-N, $R^2$ лог-лог регрессии) с явной формулой.
- [ ] **Воспроизводимость**: зафиксирован `random.seed`/`np.random.seed`; результаты повторяемы при повторном запуске.
- [ ] **Вывод**: для каждой демо-задачи и каждой самостоятельной задачи есть содержательный вывод (1–2 предложения), интерпретирующий полученные числа, а не просто их пересказ.
